In [ ]:
print("Hi")

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [2]:
# =========================
# STEP 1: RAG SETUP
# =========================
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Load data
loader = WebBaseLoader("https://docs.langchain.com/langsmith/agent-server")
data = loader.load()

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(data)

# Vector DB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# =========================
# STEP 2: RAG CHAIN
# =========================
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from context. If not found, say I don't know."),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)


# =========================
# STEP 3: TOOLS
# =========================
from langchain.tools import tool

# --- RAG Tool ---
@tool
def rag_search(question: str) -> str:
    """Search LangSmith docs and answer questions based on context."""
    return rag_chain.invoke(question)

# --- Math Tools ---
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


# =========================
# STEP 4: CREATE AGENT (MODERN)
# =========================
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[rag_search, add, multiply]
)


# =========================
# STEP 5: TEST QUERIES
# =========================
queries = [
    "What is 5 plus 7?",
    "Multiply 6 and 8",
    "Explain LangSmith deployment",
    "Multiply 3 and 4 and explain LangSmith deployment"
]

for q in queries:
    print(f"\n🧠 Question: {q}")
    
    response = agent.invoke({
        "messages": [{"role": "user", "content": q}]
    })
    
    print("✅ Answer:", response["messages"][-1].content)

USER_AGENT environment variable not set, consider setting it to identify your requests.



🧠 Question: What is 5 plus 7?
✅ Answer: 5 plus 7 equals 12.

🧠 Question: Multiply 6 and 8
✅ Answer: The result of multiplying 6 and 8 is 48.

🧠 Question: Explain LangSmith deployment
✅ Answer: LangSmith deployment involves deploying an Agent Server application, which includes specifying the graphs to deploy along with necessary configuration settings such as dependencies and environment variables. The deployment process includes:

- Deploying one or more graphs, which act as blueprints for assistants (agents designed for specific tasks).
- Setting up a database for persistence to store the application's data. While LangSmith manages the database automatically, if you're deploying on personal infrastructure, you will need to set it up independently.
- Incorporating a task queue to handle asynchronous processing tasks within the application.

Overall, the deployment ensures that the application functions as intended with all necessary components in place.

🧠 Question: Multiply 3 and 4 a